# Knife Detection — YOLOv8 Fine-Tune
**Run on Google Colab free GPU (T4)**

Steps:
1. Run all cells top to bottom
2. Upload your dataset zip when prompted
3. Wait ~20–40 min for training
4. Download `best.pt` from the last cell
5. Copy it to your project root as `hazard_yolo.pt`

## 1. Check GPU

In [ ]:
import torch
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))
else:
    print('WARNING: No GPU detected. Go to Runtime → Change runtime type → T4 GPU')

## 2. Install dependencies

In [ ]:
!pip install ultralytics --quiet

## 3. Upload dataset via Google Drive (faster than direct upload)

**Before running this cell:**
1. Go to [drive.google.com](https://drive.google.com) in your browser
2. Drag and drop `knife-dataset.zip` into My Drive (upload happens in the browser tab — much faster)
3. Wait for the Drive upload to finish, then run this cell

In [ ]:
from google.colab import drive
import zipfile, os

# Mount Google Drive
drive.mount('/content/drive')

# Find the zip in Drive (searches My Drive root first, then recursively)
ZIP_NAME = 'knife-dataset.zip'
ZIP_PATH = None
for root, dirs, files in os.walk('/content/drive/MyDrive'):
    for f in files:
        if f == ZIP_NAME:
            ZIP_PATH = os.path.join(root, f)
            break

if ZIP_PATH is None:
    raise FileNotFoundError(
        f'{ZIP_NAME} not found in Google Drive.\n'
        'Make sure you uploaded it to My Drive and the upload completed.'
    )

print(f'Found zip: {ZIP_PATH}')
print('Extracting...')
with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    z.extractall('/content/dataset')

print('Done. Files extracted to /content/dataset')
!find /content/dataset -name "data.yaml"

## 4. Fix paths in data.yaml for Colab

In [ ]:
import yaml, os

# Find data.yaml anywhere under /content/dataset
DATA_YAML = None
for root, dirs, filenames in os.walk('/content/dataset'):
    for f in filenames:
        if f == 'data.yaml':
            DATA_YAML = os.path.join(root, f)
            break

if DATA_YAML is None:
    raise FileNotFoundError('data.yaml not found under /content/dataset')

print(f'Found data.yaml: {DATA_YAML}')
dataset_root = os.path.dirname(DATA_YAML)

with open(DATA_YAML, 'r') as f:
    cfg = yaml.safe_load(f)

# Map each split to its actual folder inside the dataset directory
SPLIT_DIRS = {
    'train': os.path.join(dataset_root, 'train', 'images'),
    'val':   os.path.join(dataset_root, 'valid', 'images'),
    'test':  os.path.join(dataset_root, 'test',  'images'),
}

for key, path in SPLIT_DIRS.items():
    if key in cfg:
        cfg[key] = path

with open(DATA_YAML, 'w') as f:
    yaml.dump(cfg, f)

print('Updated config:', cfg)

# Verify
for key, path in SPLIT_DIRS.items():
    exists = os.path.exists(path)
    print(f'  {key}: {path}  → {"EXISTS ✓" if exists else "MISSING ✗"}')

## 5. Train

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8s.pt')   # downloads ~22MB base weights automatically

model.train(
    data=DATA_YAML,
    epochs=50,
    imgsz=640,
    batch=16,
    name='knife_finetune',
    patience=10,
    device=0,       # GPU
    plots=True,
    verbose=True,
)

## 6. Evaluate (optional)

In [ ]:
best_model = YOLO('runs/detect/knife_finetune/weights/best.pt')
metrics = best_model.val(data=DATA_YAML)
print(f"mAP50: {metrics.box.map50:.3f}")
print(f"mAP50-95: {metrics.box.map:.3f}")

## 7. Download best.pt

After downloading, copy it to your project root and rename to `hazard_yolo.pt`:
```
copy best.pt G:\ChildSafety-Backend-API\hazard_yolo.pt
```
Then restart the camera loop — yolo.py picks it up automatically.

In [ ]:
from google.colab import files
files.download('runs/detect/knife_finetune/weights/best.pt')